In [10]:
# Model1: XGBoost model to predict Electrical Conductance (EC)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

import optuna  # pip install optuna
# XGBoost (install first if needed: `pip install xgboost`)
import xgboost as xgb

In [11]:
# Load EC training dataset (features + target already joined)

data_path = "../../Datasets_Ours/Final Datasets/ec_training_complete.csv"
full = pd.read_csv(data_path)

print("EC Training dataset shape:", full.shape)
print("Columns:", list(full.columns))
full.head()

EC Training dataset shape: (9319, 24)
Columns: ['latitude', 'longitude', 'month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count', 'electrical_conductance']


,latitude,longitude,month_fitted,swir16,swir22,red,NDMI,MNDWI,pet,aet,...,srad,tmax,tmin,vap,vpd,ws,pdsi,esa_lccs_class,esa_change_count,electrical_conductance
0,-34.405833,19.600556,523.701326,13580.000000,10717.000000,9095.500000,0.166424,-0.184320,132.300003,11.900001,...,263.801483,23.480000,12.179999,1.372,0.79,3.48,-2.59,120.0,0.0,1008.0
1,-34.405833,19.600556,540.219951,14055.575913,11745.353685,10390.225258,0.012771,-0.164021,70.400002,67.599998,...,158.596588,17.779999,7.240000,1.004,0.53,3.36,-3.30,120.0,0.0,1237.0
2,-34.405833,19.600556,470.470642,14115.066596,11908.435926,10413.610383,0.010238,-0.173084,163.000000,18.000000,...,323.498383,26.469999,15.740000,1.703,0.93,2.07,-3.49,120.0,0.0,1053.0
3,-34.405833,19.600556,528.790206,11536.500000,9401.000000,8631.000000,0.129337,-0.139379,51.400002,43.700001,...,113.603378,17.420000,8.400000,1.106,0.45,3.50,-1.60,120.0,0.0,1167.0
4,-34.405833,19.600556,538.392487,13263.500000,10589.000000,9197.500000,0.139460,-0.176520,88.200005,58.600002,...,195.097321,19.730000,10.200000,1.231,0.55,3.64,-1.40,120.0,0.0,877.0


In [12]:
# Build feature matrix X and target y for EC

# Columns to exclude from features
exclude_cols = {
    "electrical_conductance",            # target
    "latitude", "longitude",            # spatial identifiers
}

base_feature_cols = [c for c in full.columns if c not in exclude_cols]
X_full = full[base_feature_cols].copy()
y = full["electrical_conductance"]

print("Initial number of features:", len(base_feature_cols))
print("Features:", base_feature_cols)

# 1) Remove multicollinearity: drop one of each highly correlated pair
corr_matrix = X_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_threshold = 0.95
cols_to_drop_mc = [col for col in upper.columns if any(upper[col] > high_corr_threshold)]

X_mc = X_full.drop(columns=cols_to_drop_mc)
feature_cols_mc = list(X_mc.columns)

print(f"Dropped {len(cols_to_drop_mc)} highly correlated features (>|{high_corr_threshold}|)")
print("Remaining features after multicollinearity reduction:", len(feature_cols_mc))

# For downstream cells we will further reduce by feature importance, then set X and feature_cols there.
# For now, expose X_mc and feature_cols_mc
X_reduced_mc = X_mc.copy()
feature_cols_reduced_mc = feature_cols_mc

print("Example remaining feature columns:", feature_cols_reduced_mc[:10])

Initial number of features: 21
Features: ['month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count']
Dropped 2 highly correlated features (>|0.95|)
Remaining features after multicollinearity reduction: 19
Example remaining feature columns: ['month_fitted', 'swir16', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt']


In [13]:
# Feature selection via XGBoost feature importance (on multicollinearity-reduced set)

# Use a reasonably strong but not overfitted model to rank features
fs_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

fs_model.fit(X_reduced_mc, y)

importances = fs_model.feature_importances_
fi_fs = pd.DataFrame({"feature": feature_cols_reduced_mc, "importance": importances})
fi_fs = fi_fs.sort_values("importance", ascending=False).reset_index(drop=True)
fi_fs["cum_importance"] = fi_fs["importance"].cumsum()

# Keep features that explain up to 90% of total importance, but ensure at least 20 features
importance_cutoff = 0.90
min_features = 20
selected = fi_fs[fi_fs["cum_importance"] <= importance_cutoff]["feature"].tolist()
if len(selected) < min_features:
    selected = fi_fs.head(min_features)["feature"].tolist()

X = X_reduced_mc[selected].copy()
feature_cols = selected

print("Total features after multicollinearity reduction:", len(feature_cols_reduced_mc))
print("Selected features after importance-based selection:", len(feature_cols))
print("Top selected features:", feature_cols[:15])

Total features after multicollinearity reduction: 19
Selected features after importance-based selection: 19
Top selected features: ['soil', 'esa_change_count', 'esa_lccs_class', 'vpd', 'vap', 'MNDWI', 'pdsi', 'ws', 'tmin', 'pet', 'def', 'ppt', 'NDMI', 'swir16', 'red']


In [14]:
# Train XGBoost model and report R² + feature importances

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# R² scores
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Train R²: {r2_train:.3f}")
print(f"Test  R²: {r2_test:.3f}")

# Feature importances
importances = model.feature_importances_
fi = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi = fi.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting EC:")
print(fi.head(20).to_string(index=False))

fi.head(20)

Train R²: 0.909
Test  R²: 0.702

Top 20 most important features for predicting EC:
         feature  importance
            soil    0.130285
esa_change_count    0.104395
  esa_lccs_class    0.095007
          swir16    0.058077
             vpd    0.057192
            pdsi    0.050276
           MNDWI    0.047941
             vap    0.046837
             red    0.044663
             pet    0.043950
             def    0.043705
             aet    0.039415
            NDMI    0.039384
            tmin    0.037668
              ws    0.037106
            tmax    0.034567
    month_fitted    0.033905
             ppt    0.031155
               q    0.024473


,feature,importance
0,soil,0.130285
1,esa_change_count,0.104395
2,esa_lccs_class,0.095007
13,swir16,0.058077
3,vpd,0.057192
6,pdsi,0.050276
5,MNDWI,0.047941
4,vap,0.046837
14,red,0.044663
9,pet,0.043950


In [15]:
# Stratified K-Fold + Optuna hyperparameter tuning for EC model

# Bin the continuous target into quantiles for stratification
n_bins = 10
# qcut can have duplicate bin edges; drop duplicates
y_strat = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 0.8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0),
        "gamma": trial.suggest_float("gamma", 0.5, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    cv_scores = []
    for train_idx, valid_idx in skf.split(X, y_strat):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        y_val_pred = model.predict(X_val)
        cv_scores.append(r2_score(y_val, y_val_pred))

    return float(np.mean(cv_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV R²:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-02-27 04:12:12,250] A new study created in memory with name: no-name-0f797e91-9f52-44b4-86f7-12cf1a623f1e
Best trial: 0. Best value: 0.469217:   2%|▎         | 1/40 [00:01<01:12,  1.85s/it]

[I 2026-02-27 04:12:14,098] Trial 0 finished with value: 0.46921747407455283 and parameters: {'n_estimators': 255, 'max_depth': 3, 'learning_rate': 0.04249053730836662, 'subsample': 0.5742900201906006, 'colsample_bytree': 0.5269269100859562, 'min_child_weight': 7.316503947276077, 'gamma': 4.905531624818095, 'reg_alpha': 0.5962474526940991, 'reg_lambda': 3.805088968536832}. Best is trial 0 with value: 0.46921747407455283.


Best trial: 0. Best value: 0.469217:   5%|▌         | 2/40 [00:03<01:12,  1.92s/it]

[I 2026-02-27 04:12:16,067] Trial 1 finished with value: 0.40829359274035104 and parameters: {'n_estimators': 210, 'max_depth': 4, 'learning_rate': 0.012235692824635064, 'subsample': 0.7445917209986413, 'colsample_bytree': 0.7884597837663909, 'min_child_weight': 7.130546138284976, 'gamma': 3.7702343533755176, 'reg_alpha': 0.7308100049631572, 'reg_lambda': 3.577065408942812}. Best is trial 0 with value: 0.46921747407455283.


Best trial: 2. Best value: 0.587351:   8%|▊         | 3/40 [00:06<01:23,  2.25s/it]

...

[I 2026-02-27 04:14:24,956] Trial 37 finished with value: 0.6187153577187925 and parameters: {'n_estimators': 305, 'max_depth': 5, 'learning_rate': 0.07220371951422033, 'subsample': 0.5581556420515413, 'colsample_bytree': 0.6901601238648023, 'min_child_weight': 19.14195977819389, 'gamma': 2.679720872660642, 'reg_alpha': 0.5308565421878999, 'reg_lambda': 3.7922924055957834}. Best is trial 22 with value: 0.6457437103447937.


Best trial: 22. Best value: 0.645744:  98%|█████████▊| 39/40 [02:16<00:03,  3.19s/it]

[I 2026-02-27 04:14:28,714] Trial 38 finished with value: 0.6138718287698705 and parameters: {'n_estimators': 329, 'max_depth': 5, 'learning_rate': 0.04996070376186539, 'subsample': 0.6375761009751166, 'colsample_bytree': 0.7413473070494856, 'min_child_weight': 15.428582867943192, 'gamma': 1.8012056618025034, 'reg_alpha': 0.3195715335662568, 'reg_lambda': 3.125505429161766}. Best is trial 22 with value: 0.6457437103447937.


Best trial: 22. Best value: 0.645744: 100%|██████████| 40/40 [02:19<00:00,  3.49s/it]

[I 2026-02-27 04:14:31,843] Trial 39 finished with value: 0.5037483510964623 and parameters: {'n_estimators': 286, 'max_depth': 5, 'learning_rate': 0.014913621474072351, 'subsample': 0.6845867412882305, 'colsample_bytree': 0.5439289746911153, 'min_child_weight': 18.144029684182545, 'gamma': 3.404559224738736, 'reg_alpha': 0.3968412032657098, 'reg_lambda': 3.4173786648287487}. Best is trial 22 with value: 0.6457437103447937.
Best CV R²: 0.6457437103447937
Best params:
  n_estimators: 316
  max_depth: 5
  learning_rate: 0.09904658087920225
  subsample: 0.6149812047016854
  colsample_bytree: 0.6871228967659683
  min_child_weight: 16.50715367090959
  gamma: 1.9429773430298614
  reg_alpha: 0.277037813035037
  reg_lambda: 3.2459019531177167


In [16]:
# Train final model with best hyperparameters and report R² + feature importances

best_params = study.best_params.copy()
best_params.update({"random_state": 42, "n_jobs": -1})

final_model = xgb.XGBRegressor(**best_params)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Final model (with tuned params) Train R²: {r2_train:.3f}")
print(f"Final model (with tuned params) Test  R²: {r2_test:.3f}")

# Feature importances from tuned model
importances = final_model.feature_importances_
fi_tuned = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_tuned = fi_tuned.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting EC (tuned model):")
print(fi_tuned.head(40).to_string(index=False))

fi_tuned.head(40)

Final model (with tuned params) Train R²: 0.833
Final model (with tuned params) Test  R²: 0.672

Top 20 most important features for predicting EC (tuned model):
         feature  importance
            soil    0.138058
esa_change_count    0.119260
  esa_lccs_class    0.090733
             vpd    0.056925
          swir16    0.052426
             def    0.051892
           MNDWI    0.046568
            pdsi    0.046547
             pet    0.046156
             vap    0.043661
            tmin    0.038823
            NDMI    0.038216
    month_fitted    0.037536
             aet    0.037082
             red    0.035324
              ws    0.033640
            tmax    0.030301
             ppt    0.029252
               q    0.027601


,feature,importance
0,soil,0.138058
1,esa_change_count,0.119260
2,esa_lccs_class,0.090733
3,vpd,0.056925
13,swir16,0.052426
10,def,0.051892
5,MNDWI,0.046568
6,pdsi,0.046547
9,pet,0.046156
4,vap,0.043661


In [17]:
# Create predictions for EC on validation data and update submission file

# Load submission template
sub_template = pd.read_csv("../../submission_template.csv")
print("Submission template shape:", sub_template.shape)
print("Template columns:", list(sub_template.columns))

# VALIDATION CHECK: Ensure template has exactly 200 rows
assert sub_template.shape[0] == 200, f"Template should have 200 rows, found {sub_template.shape[0]}!"

# Load validation features
val_features = pd.read_csv("../../Datasets_Ours/Final Datasets/ec_validation.csv")
print("\nValidation dataset shape:", val_features.shape)
print("Validation columns:", list(val_features.columns))

# VALIDATION CHECK: Ensure validation has exactly 200 rows
assert val_features.shape[0] == 200, f"Validation should have 200 rows, found {val_features.shape[0]}!"

# Build X for validation using the same selected feature set as training
X_val = val_features[feature_cols].copy()
print("\nValidation features shape:", X_val.shape)
print(f"Using {len(feature_cols)} features: {feature_cols[:5]}...")

# Predict EC for validation rows
ec_pred = final_model.predict(X_val)
print(f"\nGenerated {len(ec_pred)} EC predictions")
print(f"EC predictions - Min: {ec_pred.min():.2f}, Max: {ec_pred.max():.2f}, Mean: {ec_pred.mean():.2f}")

# CRITICAL: Match predictions to template by LAT/LON/DATE (not row order!)
val_features['EC_prediction'] = ec_pred

# Standardize column names for merge
val_features_std = val_features.rename(columns={
    'latitude': 'Latitude',
    'longitude': 'Longitude',
    'sample_date': 'Sample Date'
})

# Merge predictions with template by coordinates AND date
submission = sub_template.merge(
    val_features_std[['Latitude', 'Longitude', 'Sample Date', 'EC_prediction']],
    on=['Latitude', 'Longitude', 'Sample Date'],
    how='left'
)

# Rename prediction column
submission['Electrical Conductance'] = submission['EC_prediction']
submission = submission.drop(columns=['EC_prediction'])

# Ensure column order matches template
submission = submission[sub_template.columns]

# VALIDATION CHECK: Ensure no missing predictions
missing_count = submission['Electrical Conductance'].isnull().sum()
if missing_count > 0:
    print(f"\n⚠️  WARNING: {missing_count} locations without EC predictions!")
    print("Missing locations:")
    print(submission[submission['Electrical Conductance'].isnull()][['Latitude', 'Longitude', 'Sample Date']].head(10))
else:
    print("\n✓ All 200 locations have EC predictions")

# Save submission file
out_path = "../../submission1.csv"
submission.to_csv(out_path, index=False)
print(f"\nSaved submission file: {out_path}")
print("Submission shape:", submission.shape)
print("\nFirst few rows of submission:")
submission.head()

Submission template shape: (200, 6)
Template columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

Validation dataset shape: (200, 24)
Validation columns: ['latitude', 'longitude', 'sample_date', 'month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count']

Validation features shape: (200, 19)
Using 19 features: ['soil', 'esa_change_count', 'esa_lccs_class', 'vpd', 'vap']...

Generated 200 EC predictions
EC predictions - Min: 87.18, Max: 791.43, Mean: 372.91

✓ All 200 locations have EC predictions

Saved submission file: ../../submission1.csv
Submission shape: (200, 6)

First few rows of submission:


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,289.397919,NaN
1,-33.329167,26.077500,16-09-2015,NaN,440.410156,NaN
2,-32.991639,27.640028,07-05-2015,NaN,295.260529,NaN
3,-34.096389,24.439167,07-02-2012,NaN,393.329376,NaN
4,-32.000556,28.581667,01-10-2014,NaN,199.030121,NaN
